<a href="https://colab.research.google.com/github/EliasNoorzad/spacecraft-pose-estimation/blob/main/notebooks/yolo_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# YOLOv8s-Pose training

DLVS3-HST · 37 keypoints · Google Colab

Dataset: `MyDrive/hst_yolo_pose/`. Use a GPU runtime.

In [ ]:
%pip install -q ultralytics pyyaml

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
import torch
import yaml

# Read the dataset from Drive and keep the trained weights there.
drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Enable a GPU under Runtime > Change runtime type'
print('GPU:', torch.cuda.get_device_name(0))

DRIVE_DATASET = Path('/content/drive/MyDrive/hst_yolo_pose')
LOCAL_DATASET = Path('/content/hst_yolo_pose')

for relative in ('images/train', 'images/val', 'labels/train', 'labels/val', 'data.yaml'):
    assert (DRIVE_DATASET / relative).exists(), f'Missing dataset item: {DRIVE_DATASET / relative}'

# Working with a local copy is faster than loading thousands of files from Drive.
if LOCAL_DATASET.exists():
    shutil.rmtree(LOCAL_DATASET)
shutil.copytree(DRIVE_DATASET, LOCAL_DATASET)

yaml_path = LOCAL_DATASET / 'data.yaml'
with yaml_path.open('r', encoding='utf-8') as f:
    config = yaml.safe_load(f)
# The dataset YAML may contain a Windows path, so point it to the Colab copy.
config['path'] = str(LOCAL_DATASET)
config['kpt_shape'] = [37, 3]
# Keep the experimental OKS setting explicit when comparing runs.
config['kpt_oks_sigmas'] = [0.10] * 37
with yaml_path.open('w', encoding='utf-8') as f:
    yaml.safe_dump(config, f, sort_keys=False)

# Each label should have 5 box/class values and 37 (x, y, visibility) triplets.
for split in ('train', 'val', 'test'):
    label_dir = LOCAL_DATASET / 'labels' / split
    assert label_dir.is_dir(), f'Missing labels: {label_dir}'
    examples = sorted(label_dir.glob('*.txt'))
    assert examples, f'No labels in {label_dir}'
    first_line = next((line for line in examples[0].read_text().splitlines() if line.strip()), None)
    assert first_line and len(first_line.split()) == 116, f'Expected 116 values in {examples[0]}'

print('Prepared dataset:', LOCAL_DATASET)
print('Configuration:', yaml_path)

In [ ]:
from ultralytics import YOLO

# Start from pretrained pose weights rather than training from scratch.
model = YOLO('yolov8s-pose.pt')

model.train(
    data=str(yaml_path),
    epochs=100,
    imgsz=640,
    batch=8,
    device=0,
    workers=2,
    patience=25,
    seed=42,
    fliplr=0.0,  # No keypoint flip mapping is defined for HST.
    flipud=0.0,
    mosaic=0.0,
    project='/content/drive/MyDrive/HST_YOLO_Runs',  # Keep checkpoints in Drive.
    name='hst_pose_sigma_010',
    save=True,
    plots=True,
)

print('Training directory:', model.trainer.save_dir)
print('Best checkpoint:', Path(model.trainer.save_dir) / 'weights' / 'best.pt')